# 🧑‍🏫 Make Virgo smarter: new training data from Gemma 3 27B — Google Vertex AI

A bigger, smarter teacher, **Gemma 3 27B**, writes new Khmer and English training examples for Virgo
(this is called *distillation*). Each one is checked: the right language and Khmer script, a sensible
length, no "I am Gemma / Google" identity, no duplicates, and a teacher grade of 4–5 out of 5. Then
**`chat/train_vertex.ipynb` trains Virgo 12B on them** together with Virgo's own 2,003 examples.

✅ Allowed for commercial use: Gemma's terms allow using its outputs, and Virgo-1.0-Angkor is itself
a Gemma model, so nothing changes about Virgo's license. (❌ Don't do this with **Gemini**: Google's
Gemini terms forbid training competing models on its answers.)

Time and cost: about **3–6 hours** on an L4 for 3,000 examples (≈ $3–6 on-demand, less with Spot).
Progress is saved to your private Hugging Face dataset **Virgo-1.0-Angkor-Data** every 200 examples,
so if the runtime stops (Spot), just run again: it continues where it left off.

Before the first run, open [google/gemma-3-27b-it](https://huggingface.co/google/gemma-3-27b-it) on
Hugging Face and click **Agree and access**.

**First time only (Google Cloud)**
1. Create a project at [console.cloud.google.com](https://console.cloud.google.com) and turn on billing (new accounts get **$300 free credit**).
2. **IAM & Admin → Quotas**: request **1 × NVIDIA L4 GPU** (`NVIDIA_L4_GPUS`) in a region such as `us-central1`. New projects start with 0.
3. Turn on the **Vertex AI API** and the **Secret Manager API**.
4. **Security → Secret Manager → Create secret** for each secret below (name exactly as written, value = the token). Grant your Colab runtime's service account the **Secret Manager Secret Accessor** role. (Or skip this: the notebook asks for each one when it runs.)
5. **Vertex AI → Colab Enterprise → Runtime templates → New**: machine **g2-standard-8** (1 × L4, 32 GB RAM), disk 200 GB. Tick **Spot** to pay ~60–70% less (it can be stopped; just run again).

**Run**: Colab Enterprise → **Import notebook** from GitHub (`nangmrr752/virgo-models`, this file) → connect to your template's runtime → **Run all**.

**Secrets:** `HF_TOKEN` (Hugging Face token with **write** access).

In [ ]:
# ⚙️ Settings
TARGET = 3000            # examples to make in total (across runs)
KHMER_SHARE = 0.6        # 60% Khmer, 40% English
TEACHER = "google/gemma-3-27b-it"
DATA_NAME = "Virgo-1.0-Angkor-Data"   # private Hugging Face dataset the examples are saved to
OUT = "chat/data/distilled_gemma27b.jsonl"
ASK_FOR_SECRETS = True   # False for background Executions (use Secret Manager)
MODEL_NAME = "Virgo-1.0-Angkor-12B"

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: connect to a runtime template with an NVIDIA L4 (see the steps above)"
print("✅ GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
WORK = "/content" if os.path.isdir("/content") else os.path.expanduser("~")
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"])
    assert r.returncode == 0, "❌ Can't download the Virgo code from GitHub"
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-cloud-secret-manager"])
def secret(name):
    """A secret: from the environment, else Google Secret Manager (this project), else asked once."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        import google.auth
        from google.cloud import secretmanager
        _, project = google.auth.default()
        value = secretmanager.SecretManagerServiceClient().access_secret_version(
            name=f"projects/{project}/secrets/{name}/versions/latest").payload.data.decode().strip()
        os.environ[name] = value
        return value
    except Exception:
        pass
    if ASK_FOR_SECRETS:
        from getpass import getpass
        value = getpass(f"{name} (leave empty to skip): ").strip()
        if value:
            os.environ[name] = value
        return value or None
    return None

from huggingface_hub import login, HfApi
token = secret("HF_TOKEN")
assert token, "❌ No HF_TOKEN: add it in Secret Manager (see the steps above) or type it when asked"
login(token=token)
ME = HfApi().whoami()["name"]
DATA_REPO = f"{ME}/{DATA_NAME}"
print("✅ Ready. New examples will be saved to the private dataset", DATA_REPO)

def run(cmd):
    """Runs a step and shows its output here; on failure, shows the last lines (send them to Claude)."""
    import subprocess, collections
    tail = collections.deque(maxlen=60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        tail.append(line)
    if proc.wait() != 0:
        print("\n" + "=" * 60 + "\n❌ It stopped with an error. Send these last lines to Claude:\n" + "=" * 60)
        print("".join(tail))
        raise SystemExit(f"❌ {next((a for a in cmd if a.endswith('.py')), cmd[0])} failed (exit code {proc.returncode})")

In [ ]:
# 2️⃣ Continue from earlier runs (downloads the examples already made)
from huggingface_hub import hf_hub_download
try:
    path = hf_hub_download(DATA_REPO, os.path.basename(OUT), repo_type="dataset", local_dir="chat/data")
    print("⬇️ Continuing from", sum(1 for _ in open(OUT, encoding="utf-8")), "examples made earlier")
except Exception:
    print("Starting fresh")

In [ ]:
# 3️⃣ Make the examples with Gemma 3 27B (shows progress; saved to Hugging Face every 200)
import sys
run([sys.executable, "-u", "scripts/distill_gemma.py", "--teacher", TEACHER, "--target", str(TARGET),
     "--khmer-share", str(KHMER_SHARE), "--out", OUT, "--hf-repo", DATA_REPO])
run([sys.executable, "scripts/check_data.py"])

In [ ]:
# 4️⃣ A few of the new examples (send some to Claude if anything looks off)
import json, random
rows = [json.loads(l) for l in open(OUT, encoding="utf-8")]
for r in random.sample(rows, min(8, len(rows))):
    m = r["messages"]
    print("🧑", m[1]["content"][:300]); print("🤖", m[2]["content"][:500], "\n")
print(f"✅ {len(rows)} examples in {DATA_REPO}. Next: run chat/train_vertex.ipynb — it uses them automatically.")
print("Stop or delete this runtime now so it doesn't keep costing money.")